# Foundation Models for EHR and Physiological Time Series

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week08_ehr_time_series.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 8 · Reference notebook (ungraded)**

This notebook follows the Week 8 lecture and its three clips. It builds a small version of ETHOS (Renc et al., npj Digital
Medicine 2024) from scratch: a tokenizer that turns a hospital timeline into tokens, a tiny decoder trained with the
next-token loss, and zero-shot prediction by simulating 20 futures per patient. It then shows three ways to handle irregular
time, and how a dense signal (an ECG) becomes patches.

| Section | Clip | What you will see |
|---|---|---|
| 1 | A patient timeline as tokens | The clip's patient as 19 tokens: codes split into parts, quantile tokens, interval tokens, static tokens; a tiny decoder |
| 2 | Zero-shot prediction by simulating the future | 20 rollouts per patient turned into a probability; AUC on a test set; false alarms at low prevalence |
| 3 | Irregular sampling and physiological signals | An hourly grid with a mask, interval tokens, a time embedding, and ECG patches at 500 Hz |
| 4 | Optional | The same tokenizer on the open MIMIC-IV Clinical Database Demo |

**How to run it**
- **Google Colab or HiPerGator:** *Runtime > Run all*. Everything runs on a CPU in about a minute. Sections 1 to 3 need no
  download.
- **Data.** Sections 1 to 3 use **synthetic patients** generated in code: no real patient data. Section 4 is optional and
  switched off by default; it downloads the MIMIC-IV Clinical Database Demo (100 de-identified patients, open access) at run
  time. Do not save or commit those files anywhere else.

**Credits.** The tokenization rules (codes split into parts, 10 quantiles from the training data, interval tokens with none
for gaps under 5 minutes, 6 static tokens, a 2,048-token context, and 20 simulated trajectories) follow the description in
Renc et al. The code is written for this course; it is not the ETHOS code.

## 0. Setup

In [ ]:
import math, os, time, gzip, urllib.request                  # standard library
import numpy as np                                           # arrays
import torch                                                 # tensors and autograd
import torch.nn as nn                                        # layers
import torch.nn.functional as F                              # losses
import matplotlib.pyplot as plt                              # plots
from sklearn.metrics import roc_auc_score                    # AUC in section 2

torch.manual_seed(0)                                         # same results on every run
print("PyTorch", torch.__version__)

## 1. A patient timeline as tokens  *(clip 1)*

### 1.1 A hospital record is a timeline of events at irregular times
The patient from the clip, in hours after arrival in the emergency department (ED). This patient is invented.

In [ ]:
clip_patient = dict(sex="F", age=67, marital="MARRIED", bmi=27.0, start_year=2026,
                    events=[(0.0, "ED_ADMIT", None),            # (hours, event, value)
                            (0.6, "LACTATE", 4.1),               # mmol/L
                            (1.5, "norepinephrine", None),
                            (2.2, "sepsis", None),
                            (3.5, "ICU_ADMIT", None)])
plt.figure(figsize=(9, 1.6))
for t, name, v in clip_patient["events"]:
    plt.plot(t, 0, "ko"); plt.text(t, 0.15, name if v is None else f"{name} {v}", rotation=20, fontsize=8)
plt.yticks([]); plt.ylim(-0.3, 1); plt.xlabel("hours"); plt.title("one synthetic patient", fontsize=9); plt.show()

### 1.2 Codes become tokens: ETHOS splits each code into its parts
Diagnoses use ICD-10-CM codes and drugs use ATC codes. ETHOS writes a code as up to three tokens: the first three characters,
then the next two, then the rest. Related codes then share their first token, so the model can learn what they have in common.

In [ ]:
CODES = {"sepsis": "A41.9", "pneumonia": "J18.9",            # ICD-10-CM diagnoses
         "norepinephrine": "C01CA03", "ceftriaxone": "J01DD04"}   # ATC drugs

def split_code(code):
    # 'A41.9' -> ['A41', '9'];  'C01CA03' -> ['C01', 'CA', '03']
    c = code.replace(".", "")                                # ICD codes are written with a dot after 3 characters
    return [part for part in (c[:3], c[3:5], c[5:]) if part]

for name, code in CODES.items():
    print(f"{name:>15s}  {code:>8s}  ->  {split_code(code)}")

### 1.3 Numbers become quantile tokens
Each lab is cut into **10 bins with equal counts**, computed on the training data only. A value becomes two tokens: the lab
name and its bin, Q1 (lowest tenth) to Q10 (highest tenth). To compute the bins we need training data, so here is a generator
of synthetic ICU stays. Each patient has a hidden severity that raises lactate and heart rate, makes norepinephrine more
likely, and raises the risk of dying in the ICU. The labels exist only because the data are synthetic; training never uses them.

In [ ]:
def synthetic_stay(rng):
    # One synthetic ICU stay: static data, timed events (hours), and whether the patient died in the ICU.
    sev = rng.normal()                                       # hidden severity
    age = int(np.clip(rng.normal(62, 15), 20, 94))
    p = dict(sex=rng.choice(["F", "M"]).item(), age=age, marital=rng.choice(["MARRIED", "SINGLE", "WIDOWED"]).item(),
             bmi=float(np.clip(rng.normal(28, 5), 16, 50)), start_year=int(rng.choice([2024, 2026])))
    ev = [(0.0, "ED_ADMIT", None)]
    t = rng.uniform(0.2, 1.0); ev.append((t, "LACTATE", float(np.exp(0.6 + 0.35 * sev + 0.25 * rng.normal()))))
    if rng.random() < 0.5:                                   # heart rate charted in the ED for half of the patients
        t += rng.uniform(0.1, 0.8); ev.append((t, "HR", float(95 + 12 * sev + 8 * rng.normal())))
    if sev + 0.5 * rng.normal() > 0.4:                       # sicker patients get a vasopressor
        t += rng.uniform(0.2, 1.0); ev.append((t, "norepinephrine", None))
    t += rng.uniform(0.3, 1.5); ev.append((t, "sepsis" if rng.random() < 0.6 else "pneumonia", None))
    if rng.random() < 0.5:                                   # antibiotic charted before ICU transfer for half
        t += rng.uniform(0.1, 0.6); ev.append((t, "ceftriaxone", None))
    t += rng.uniform(0.5, 3.0); ev.append((t, "ICU_ADMIT", None))
    risk = 1 / (1 + math.exp(-(-3.4 + 1.6 * sev + 0.035 * (age - 62))))   # true probability of dying in the ICU
    died = rng.random() < risk
    s = sev
    for _ in range(int(rng.integers(2, 6))):                 # the ICU course: repeated lactate and heart rate
        t += rng.uniform(1, 10)
        s += (0.35 if died else -0.35) + 0.3 * rng.normal()  # patients who will die get worse
        ev.append((t, "LACTATE", float(np.exp(0.6 + 0.35 * s + 0.25 * rng.normal()))))
        t += rng.uniform(0.05, 2); ev.append((t, "HR", float(95 + 12 * s + 8 * rng.normal())))
    t += rng.uniform(2, 48); ev.append((t, "DEATH" if died else "DISCHARGE", None))
    p.update(events=ev, died=died, risk=risk)
    return p

rng = np.random.default_rng(0)
stays = [synthetic_stay(rng) for _ in range(12000)]
train, test = stays[:10000], stays[10000:]                   # 10,000 to train, 2,000 to test
print(f"{len(train)} training stays, {len(test)} test stays; ICU mortality {np.mean([p['died'] for p in stays]):.1%}")

In [ ]:
LABS = ["LACTATE", "HR"]
values = {lab: [v for p in train for (_, n, v) in p["events"] if n == lab] for lab in LABS}
CUTS = {lab: np.quantile(values[lab], np.linspace(0.1, 0.9, 9)) for lab in LABS}   # 9 cut points -> 10 bins
CUTS["BMI"] = np.quantile([p["bmi"] for p in train], np.linspace(0.1, 0.9, 9))

def quantile_token(lab, v):
    return f"Q{int(np.searchsorted(CUTS[lab], v)) + 1}"     # Q1 ... Q10

print("lactate cut points (mmol/L):", CUTS["LACTATE"].round(2))
print("lactate 4.1 ->", ["LACTATE", quantile_token("LACTATE", 4.1)])   # above the 90th percentile: Q10

### 1.4 Time gaps become interval tokens
ETHOS uses 13 interval tokens and adds none for gaps under 5 minutes. Its five shortest are 5m-15m, 15m-1h, 1h-2h, 2h-6h,
and 6h-12h; we add eight longer ones of our own to make 13.

In [ ]:
BINS = [(5 / 60, "5m-15m"), (0.25, "15m-1h"), (1, "1h-2h"), (2, "2h-6h"), (6, "6h-12h"),   # as in ETHOS
        (12, "12h-1d"), (24, "1d-3d"), (72, "3d-1w"), (168, "1w-2w"), (336, "2w-1mt"),     # our longer bins
        (720, "1mt-3mt"), (2160, "3mt-6mt"), (4320, "=6mt")]
assert len(BINS) == 13

def interval_tokens(hours):
    # Gap in hours -> [] (under 5 minutes) or [one interval token].
    if hours < 5 / 60: return []
    return [[name for lo, name in BINS if hours >= lo][-1]]

for gap in [0.6, 0.9, 0.7, 1.3, 0.05]:
    print(f"{gap:>4} h -> {interval_tokens(gap)}")

### 1.5 Static tokens come first, and the whole record becomes one sequence
ETHOS starts every timeline with 6 static tokens (sex, marital status, race, BMI, birth date as age, and the start of the
timeline). Our synthetic data have no race, so that token is always `RACE_UNKNOWN`. The clip's patient becomes **19 tokens**.

In [ ]:
def static_tokens(p):
    a = 5 * (p["age"] // 5)
    return [f"SEX_{p['sex']}", f"AGE_{a}-{a + 5}", f"MARITAL_{p['marital']}", "RACE_UNKNOWN",
            f"BMI_{quantile_token('BMI', p['bmi'])}", f"START_{p['start_year']}"]

def tokenize(p):
    # A stay -> a list of tokens, following the ETHOS rules above.
    tokens, last = static_tokens(p), None
    for t, name, v in p["events"]:
        if last is not None: tokens += interval_tokens(t - last)
        last = t
        if name in LABS: tokens += [name, quantile_token(name, v)]
        elif name in CODES: tokens += split_code(CODES[name])
        else: tokens.append(name)
    return tokens

clip_tokens = tokenize(clip_patient)
print(len(clip_tokens), "tokens:", " ".join(clip_tokens))

### 1.6 A decoder learns to predict the next token, as in Weeks 3 to 5
A tiny decoder transformer (2 layers, 96 numbers per token) trained with the next-token loss on the 10,000 training timelines.
ETHOS used the same loss on 241,015 MIMIC-IV patients (training and validation) with a 2,048-token context; our timelines
are short, so a context of 64 tokens is enough.

In [ ]:
seqs = [tokenize(p) for p in stays]
VOCAB = sorted({t for s in seqs for t in s} | set(clip_tokens))
stoi = {t: i for i, t in enumerate(VOCAB)}
PAD, CONTEXT = len(VOCAB), 64
X = torch.full((len(seqs), CONTEXT), PAD)
for i, s in enumerate(seqs):
    X[i, :len(s)] = torch.tensor([stoi[t] for t in s][:CONTEXT])
print("vocabulary:", len(VOCAB), "tokens | longest timeline:", max(len(s) for s in seqs), "tokens")

class Decoder(nn.Module):
    def __init__(self, vocab, d=96, layers=2, heads=4, context=CONTEXT):
        super().__init__()
        self.tok = nn.Embedding(vocab + 1, d)                # +1 for padding
        self.pos = nn.Embedding(context, d)                  # learned positions
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.out = nn.Linear(d, vocab)                       # scores for the next token

    def forward(self, x):
        n = x.shape[1]
        causal = torch.triu(torch.full((n, n), float("-inf")), diagonal=1)   # no peeking at later tokens
        return self.out(self.blocks(self.tok(x) + self.pos(torch.arange(n)), mask=causal, is_causal=True))

torch.manual_seed(0)
model = Decoder(len(VOCAB))
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
Xtr = X[:len(train)]
t0 = time.time()
for epoch in range(6):
    perm, total = torch.randperm(len(Xtr)), 0.0
    for i in range(0, len(Xtr), 128):
        xb = Xtr[perm[i:i + 128]]
        logits = model(xb[:, :-1])                           # predict token k+1 from tokens 1..k
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), xb[:, 1:].reshape(-1), ignore_index=PAD)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch + 1}: next-token loss {total / math.ceil(len(Xtr) / 128):.3f}")
print(f"{time.time() - t0:.0f} s")
model.eval();

## 2. Zero-shot prediction by simulating the future  *(clip 2)*

### 2.1 and 2.2 From the ICU transfer, sample one token at a time until DISCHARGE or DEATH
No labels and no extra model: we give the decoder the timeline up to `ICU_ADMIT` and let it write the rest. Each new token is
sampled from the softmax, appended, and the context slides once it is full (ETHOS: 2,048 tokens; here: 64).

In [ ]:
END = {stoi["DEATH"]: "DEATH", stoi["DISCHARGE"]: "DISCHARGE"}

@torch.no_grad()
def rollouts(prefix, n=20, max_new=60, seed=0):
    # n sampled futures of a token prefix; returns the generated token lists and how each ended.
    g = torch.Generator().manual_seed(seed)
    x = torch.tensor([stoi[t] for t in prefix]).repeat(n, 1)
    gen, ended = [[] for _ in range(n)], [None] * n
    for _ in range(max_new):
        logits = model(x[:, -CONTEXT:])[:, -1]               # sliding window: the last CONTEXT tokens
        nxt = torch.multinomial(torch.softmax(logits, dim=-1), 1, generator=g)
        x = torch.cat([x, nxt], dim=1)
        for k in range(n):
            if ended[k] is None:
                gen[k].append(VOCAB[int(nxt[k])])
                if int(nxt[k]) in END: ended[k] = END[int(nxt[k])]
        if all(e is not None for e in ended): break
    return gen, ended

prefix = clip_tokens                                         # ends with ICU_ADMIT
gen, ended = rollouts(prefix)
for k, (g, e) in enumerate(zip(gen, ended)):
    print(f"{k + 1:>2}  {' '.join(g[:8])} ... {e}  ({len(g)} tokens)")

### 2.3 and 2.4 Count how many futures end in DEATH
The estimate is the fraction of the 20 futures that end in death. With 20 rollouts it can only take 21 values: 0, 0.05, ..., 1.
The clip's 7 of 20 was illustrative; this tiny model gives its own answer.

In [ ]:
deaths = sum(e == "DEATH" for e in ended)
print(f"{deaths} of 20 futures end in DEATH -> estimated ICU mortality {deaths / 20:.2f}")
print("unfinished (no end token within the limit):", sum(e is None for e in ended))

### 2.5 How good is the estimate?
We repeat this for 500 test patients and compute the AUC. Because the data are synthetic, we also know each patient's true
risk, so we can see how close the tiny model gets to the best possible ranking. The same rollouts also give the ICU length of
stay: add up the interval tokens until the end token (we use the middle of each bin).

In [ ]:
MID = {"5m-15m": 0.17, "15m-1h": 0.6, "1h-2h": 1.5, "2h-6h": 4, "6h-12h": 9, "12h-1d": 18, "1d-3d": 48,
       "3d-1w": 120, "1w-2w": 252, "2w-1mt": 528, "1mt-3mt": 1440, "3mt-6mt": 3240, "=6mt": 4320}   # hours
t0 = time.time()
est, truth, risk, los_pred, los_true = [], [], [], [], []
for j, p in enumerate(test[:500]):
    toks = tokenize(p)
    gen, ended = rollouts(toks[:toks.index("ICU_ADMIT") + 1], seed=j)
    est.append(sum(e == "DEATH" for e in ended) / 20)
    truth.append(p["died"]); risk.append(p["risk"])
    los_pred.append(np.mean([sum(MID.get(t, 0) for t in g) for g in gen]) / 24)
    icu_t = next(t for t, n, _ in p["events"] if n == "ICU_ADMIT")
    los_true.append((p["events"][-1][0] - icu_t) / 24)
print(f"500 test patients, ICU mortality {np.mean(truth):.1%}  ({time.time() - t0:.0f} s)")
print(f"AUC of the 20-rollout estimate:     {roc_auc_score(truth, est):.3f}")
print(f"AUC of the true risk (best possible): {roc_auc_score(truth, risk):.3f}")
print(f"ICU length of stay, mean absolute error: {np.mean(np.abs(np.array(los_pred) - np.array(los_true))):.2f} days")

### 2.6 At low prevalence, a high AUC can still mean many false alarms
Flag the 20% of patients with the highest estimated risk and count what happens.

In [ ]:
est_a, truth_a = np.array(est), np.array(truth)
order = np.argsort(-est_a + 1e-9 * np.random.default_rng(0).random(len(est_a)))   # break ties at random
flagged = np.zeros(len(est_a), bool); flagged[order[: len(est_a) // 5]] = True
tp = int((flagged & truth_a).sum())
print(f"flagged {flagged.sum()} of {len(est_a)}; deaths caught {tp} of {truth_a.sum()}; "
      f"false alarms {flagged.sum() - tp} of {flagged.sum()} ({(flagged.sum() - tp) / flagged.sum():.0%})")

### 2.7 Read the results with care
ETHOS reports an ICU mortality AUC of 0.927 at 7.0% prevalence and 0.921 for hospital mortality at 2.0%. Its baselines are
numbers reported by other studies with their own exclusion criteria; MIMIC-IV comes from one hospital; and long, dense records
lose their earliest history in the 2,048-token window. Wornow et al. (the other Thursday paper) reviewed 84 EHR models and
found most were trained on MIMIC-III or PubMed and tested on tasks that say little about value to a health system.

## 3. Irregular sampling and physiological signals  *(clip 3)*

### 3.1 Measurements arrive at irregular times
The ICU day from the clip, in hours after 6 p.m.: heart rate about hourly with a gap overnight, lactate three times.

In [ ]:
HR_T = [0, 1, 2, 3, 4, 5, 8, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
HR_V = [118, 112, 108, 104, 101, 99, 110, 96, 95, 94, 94, 93, 92, 92, 91, 90, 90, 89, 89, 88]   # illustrative
LAC_T, LAC_V = [0.6, 8, 14], [4.1, 3.2, 2.0]                 # 6:36 p.m., 2 a.m., 8 a.m.
fig, ax = plt.subplots(2, 1, figsize=(9, 2.8), sharex=True)
ax[0].plot(HR_T, HR_V, "o-", ms=4); ax[0].set_ylabel("HR")
ax[1].plot(LAC_T, LAC_V, "o", color="teal"); ax[1].set_ylabel("lactate")
ax[1].set_xticks(range(0, 25, 4), ["6 p.m.", "10 p.m.", "2 a.m.", "6 a.m.", "10 a.m.", "2 p.m.", "6 p.m."])
plt.tight_layout(); plt.show()

### 3.2 and 3.3 Option 1: an hourly grid, filled, with a mask
Put each value in its hour, fill empty hours with the last measured value, and keep a mask of 1 (measured) and 0 (filled).
The mask matters: a lactate drawn at 2 a.m. tells the model that someone was worried.

In [ ]:
def hourly_grid(times, vals, hours=12):
    grid, mask, last = np.full(hours, np.nan), np.zeros(hours, int), np.nan
    for h in range(hours):
        here = [v for t, v in zip(times, vals) if h <= t < h + 1]
        if here: last, mask[h] = here[-1], 1
        grid[h] = last                                       # forward fill
    return grid, mask

hr_grid, hr_mask = hourly_grid(HR_T, HR_V)
lac_grid, lac_mask = hourly_grid(LAC_T, LAC_V)
print("hour      ", " ".join(f"{h:>4d}" for h in range(12)))
print("HR        ", " ".join(f"{v:>4.0f}" for v in hr_grid))
print("measured? ", " ".join(f"{m:>4d}" for m in hr_mask))
print("lactate   ", " ".join(f"{v:>4.1f}" for v in lac_grid))
print("measured? ", " ".join(f"{m:>4d}" for m in lac_mask), "  <- hour 8 is 2 a.m.")

### 3.4 Option 2: interval tokens
The same day as tokens: the overnight gap becomes a token the model can read.

In [ ]:
events = sorted([(t, "HR", v) for t, v in zip(HR_T, HR_V)] + [(t, "LACTATE", v) for t, v in zip(LAC_T, LAC_V)])
tokens, last = [], None
for t, name, v in events[:10]:
    if last is not None: tokens += interval_tokens(t - last)
    tokens += [name, quantile_token(name, v)]; last = t
print(" ".join(tokens))

### 3.5 Option 3: a time embedding in real hours
Like the Week 2 position waves, but the input is the time in hours and each pair (sine, cosine) has a period in hours.
At t = 2.5 h, the 8-hour pair is (0.92, −0.38) and the 24-hour pair is (0.61, 0.79), as in the clip.

In [ ]:
def time_embedding(hours, periods=(8, 24)):
    return np.array([f(2 * np.pi * hours / P) for P in periods for f in (np.sin, np.cos)])

print("t = 2.5 h ->", time_embedding(2.5).round(2))          # [ 0.92 -0.38  0.61  0.79]

### 3.6 Dense signals are cut into patches, as in Week 6
A 10-second ECG at **500 Hz** is 5,000 samples, cut into 10 one-second patches of 500 samples: the tokens of an ECG
foundation model. (The Week 6 notebook used 250 Hz only so that it would run faster.) The generator is the same as in Week 6.

In [ ]:
FS = 500                                                     # samples per second

def ecg(hr=72, af=False, seed=0, dur=10):
    # Single-lead synthetic ECG: the generator from the Week 6 clips and notebook.
    rng = np.random.default_rng(seed); beats = [rng.uniform(0.25, 0.55)]
    while beats[-1] < dur + 1:
        beats.append(beats[-1] + (rng.uniform(0.42, 0.95) if af else 60 / hr * rng.uniform(0.97, 1.03)))
    t = np.arange(FS * dur) / FS; out = np.zeros_like(t)
    bump = lambda c, a, w: a * np.exp(-0.5 * ((t - c) / w) ** 2)
    for c in beats:
        if not af: out += bump(c - 0.17, 0.13, 0.025)        # P wave
        out += bump(c - 0.03, -0.12, 0.009) + bump(c, 1.0, 0.011) + bump(c + 0.035, -0.28, 0.011) + bump(c + 0.28, 0.27, 0.045)
    if af:
        ph = rng.uniform(0, 6.28, 2)
        out += 0.045 * np.sin(2 * np.pi * 6.3 * t + ph[0]) + 0.03 * np.sin(2 * np.pi * 8.9 * t + ph[1])
    return out

signal = ecg(hr=96, seed=5)
patches = signal.reshape(10, FS)                             # (10 patches, 500 samples)
print("samples:", signal.shape[0], "-> patches:", patches.shape)
fig, axes = plt.subplots(1, 10, figsize=(12, 1.4), sharey=True)
for a, p in zip(axes, patches): a.plot(p, lw=0.8); a.axis("off")
plt.suptitle("10 one-second patches of 500 samples", fontsize=9); plt.show()

## 4. Optional: the same tokenizer on the MIMIC-IV Clinical Database Demo
The demo holds 100 de-identified patients from MIMIC-IV and is open access on PhysioNet, with no credentialing, under the Open
Database License. It is real patient data, so the notebook downloads it at run time into your own folder; do not commit or
share copies. Set `RUN_MIMIC_DEMO = True` to run this section.

*This section contains information from the MIMIC-IV Clinical Database Demo (version 2.2), which is made available under the
Open Database License v1.0 (https://opendatacommons.org/licenses/odbl/1-0/). Johnson, A., Bulgarelli, L., Pollard, T.,
Horng, S., Celi, L. A., & Mark, R. (2023). MIMIC-IV Clinical Database Demo (version 2.2). PhysioNet.
https://doi.org/10.13026/dp1f-ex47*

In [ ]:
RUN_MIMIC_DEMO = False
if RUN_MIMIC_DEMO:
    import pandas as pd
    DEMO = os.path.expanduser("~/bme6938_data/mimic-iv-demo-2.2")   # on HiPerGator, use a folder on /blue
    os.makedirs(DEMO, exist_ok=True)
    for f in ["d_labitems.csv.gz", "labevents.csv.gz"]:
        path = os.path.join(DEMO, f)
        if not os.path.exists(path):
            urllib.request.urlretrieve(f"https://physionet.org/files/mimic-iv-demo/2.2/hosp/{f}", path)
    items = pd.read_csv(os.path.join(DEMO, "d_labitems.csv.gz"))
    lactate_ids = items.loc[items["label"].str.lower() == "lactate", "itemid"].tolist()
    labs = pd.read_csv(os.path.join(DEMO, "labevents.csv.gz"), usecols=["subject_id", "hadm_id", "itemid", "charttime", "valuenum"])
    lac = labs[labs["itemid"].isin(lactate_ids)].dropna(subset=["valuenum", "hadm_id"]).copy()
    lac["charttime"] = pd.to_datetime(lac["charttime"])
    cuts = np.quantile(lac["valuenum"], np.linspace(0.1, 0.9, 9))   # 10 bins from the demo itself
    print("lactate item ids:", lactate_ids, "| measurements:", len(lac), "| cut points:", cuts.round(1))
    adm = lac.groupby("hadm_id").size().idxmax()             # the admission with the most lactates
    one = lac[lac["hadm_id"] == adm].sort_values("charttime")
    tokens, last = [], None
    for t, v in zip(one["charttime"], one["valuenum"]):
        if last is not None: tokens += interval_tokens((t - last).total_seconds() / 3600)
        tokens += ["LACTATE", f"Q{int(np.searchsorted(cuts, v)) + 1}"]; last = t
    print(f"admission with {len(one)} lactates -> {len(tokens)} tokens:")
    print(" ".join(tokens[:40]), "..." if len(tokens) > 40 else "")
    print("\nContains information from the MIMIC-IV Clinical Database Demo (v2.2), available under the ODbL v1.0.")
else:
    print("Skipped. Set RUN_MIMIC_DEMO = True to download the MIMIC-IV demo (about 2 MB) and tokenize its lactates.")

## Recap

| Clip | Code in this notebook |
|---|---|
| 1. A patient timeline as tokens | `split_code`, `quantile_token`, `interval_tokens`, `static_tokens`, `tokenize` (19 tokens), a tiny `Decoder` |
| 2. Zero-shot prediction | `rollouts`: 20 futures per patient, AUC against the true risk, length of stay, false alarms |
| 3. Irregular sampling and signals | `hourly_grid` with a mask, interval tokens, `time_embedding`, ECG patches at 500 Hz |

> **Try it (optional):** use 5 or 100 rollouts in section 2.5 and watch the AUC and the number of possible values change. Use 5
> quantiles instead of 10. Train the decoder for 2 epochs instead of 6 and compare the AUC.